# 监督学习与无监督学习如何区分？

**面试回答主线：**监督学习拥有预测时刻可获得的标签，目标是最小化输入到标签的泛化误差；无监督学习只观察输入，目标是发现人为声明的结构。区别不在算法名称，而在优化目标和评估信号。本实验使用同一批外卖商户画像：先用“是否在晚高峰超时”做监督分类，再不看标签做运营分群。小数据只验证机制，不能宣称线上收益。

## 真实案例

离线的 8 家商户记录包含平均备餐分钟、骑手等待分钟和高峰超时标签。标签在订单完结后获得，因而可用于训练；分群阶段刻意不读取标签，以模拟新商户的运营画像分析。

In [1]:
import numpy as np  # 导入 NumPy 以手写数值计算。
np.set_printoptions(precision=3, suppress=True)  # 让中间数值更容易阅读。
merchant_names = np.array(['清汤面', '快炒饭', '轻食沙拉', '火锅外卖', '粥铺', '炸鸡店', '咖啡店', '披萨店'])  # 构造可读的商户名称。
features = np.array([[8, 2], [19, 7], [6, 1], [24, 9], [10, 3], [18, 6], [5, 1], [22, 8]], dtype=float)  # 每行依次是备餐分钟与骑手等待分钟。
late_label = np.array([0, 1, 0, 1, 0, 1, 0, 1], dtype=float)  # 构造晚高峰是否超时的已知标签。
print('教学实验：商户 | 备餐分钟 | 等待分钟 | 超时标签')  # 输出字段含义。
for name, row, label in zip(merchant_names, features, late_label):  # 逐条展示带业务语义的原始样本。
    print(f'{name:4s} | {row[0]:4.0f} | {row[1]:4.0f} | {int(label)}')  # 输出一条商户记录。
print('特征形状:', features.shape)  # 输出矩阵形状以确认样本与特征维度。

教学实验：商户 | 备餐分钟 | 等待分钟 | 超时标签
清汤面  |    8 |    2 | 0
快炒饭  |   19 |    7 | 1
轻食沙拉 |    6 |    1 | 0
火锅外卖 |   24 |    9 | 1
粥铺   |   10 |    3 | 0
炸鸡店  |   18 |    6 | 1
咖啡店  |    5 |    1 | 0
披萨店  |   22 |    8 | 1
特征形状: (8, 2)


## Baseline / 基线

先使用业务规则：备餐时间超过 15 分钟就预测超时。它不需要训练，也没有利用骑手等待这个特征。

In [2]:
baseline_pred = (features[:, 0] > 15).astype(float)  # 按备餐时间阈值产生规则基线预测。
baseline_accuracy = float(np.mean(baseline_pred == late_label))  # 计算基线预测的准确率。
print('规则基线预测:', baseline_pred.astype(int))  # 展示每个商户的基线结果。
print(f'规则基线准确率: {baseline_accuracy:.3f}')  # 输出可比较的基线指标。

规则基线预测: [0 1 0 1 0 1 0 1]
规则基线准确率: 1.000


In [3]:
mean = features.mean(axis=0)  # 只在教学训练样本上计算特征均值。
std = features.std(axis=0)  # 只在教学训练样本上计算特征标准差。
x = (features - mean) / std  # 标准化输入避免大尺度特征主导梯度。
x = np.c_[np.ones(len(x)), x]  # 添加截距列以手写线性得分。
w = np.zeros(x.shape[1])  # 将逻辑回归权重初始化为零。
for step in range(240):  # 迭代执行小批量梯度下降。
    score = x @ w  # 计算每条订单的线性风险得分。
    probability = 1.0 / (1.0 + np.exp(-score))  # 用 sigmoid 将得分映射为超时概率。
    gradient = x.T @ (probability - late_label) / len(x)  # 根据交叉熵推导计算平均梯度。
    w -= 0.25 * gradient  # 沿负梯度更新模型权重。
final_probability = 1.0 / (1.0 + np.exp(-(x @ w)))  # 计算训练后的最终概率。
supervised_pred = (final_probability >= 0.5).astype(float)  # 按固定阈值转成监督分类结果。
supervised_accuracy = float(np.mean(supervised_pred == late_label))  # 计算监督模型在这组受控数据上的准确率。
print('逻辑回归权重 [截距, 备餐, 等待]:', w)  # 展示学习出的可解释参数。
print('监督概率:', np.round(final_probability, 3))  # 展示标签驱动学习后的中间概率。
print(f'监督准确率: {supervised_accuracy:.3f}')  # 输出监督学习结果。

逻辑回归权重 [截距, 备餐, 等待]: [0.096 2.665 2.577]
监督概率: [0.012 0.982 0.003 0.999 0.058 0.941 0.002 0.997]
监督准确率: 1.000


In [4]:
centroids = features[[0, 3]].copy()  # 从两个差异明显的商户初始化两个聚类中心。
for step in range(8):  # 重复执行分配和更新以手写 KMeans。
    distance = ((features[:, None, :] - centroids[None, :, :]) ** 2).sum(axis=2)  # 计算每家商户到每个中心的平方距离。
    cluster = distance.argmin(axis=1)  # 选择距离最近的中心作为无监督簇编号。
    for group in range(2):  # 逐个更新两个簇的中心。
        centroids[group] = features[cluster == group].mean(axis=0)  # 用当前簇内样本均值更新中心。
print('无监督簇编号:', cluster)  # 输出未读取标签得到的簇分配。
print('无监督中心 [备餐, 等待]:', np.round(centroids, 2))  # 输出聚类假设下的画像中心。
for group in range(2):  # 解释每个簇中标签比例但不把它当训练信号。
    member_names = merchant_names[cluster == group]  # 取出当前簇的商户名称。
    late_rate = late_label[cluster == group].mean()  # 事后计算该簇的超时比例用于业务验证。
    print(f'簇 {group}: {member_names.tolist()}，事后超时率={late_rate:.2f}')  # 输出可供运营命名的分群结果。

无监督簇编号: [0 1 0 1 0 1 0 1]
无监督中心 [备餐, 等待]: [[ 7.25  1.75]
 [20.75  7.5 ]]
簇 0: ['清汤面', '轻食沙拉', '粥铺', '咖啡店']，事后超时率=0.00
簇 1: ['快炒饭', '火锅外卖', '炸鸡店', '披萨店']，事后超时率=1.00


## 结果解读

监督模型通过标签学习到备餐和等待共同增大风险；KMeans 没有任何标签，输出的只是两个几何簇。簇的事后超时率可以帮助解释，但若据此调簇数再汇报标签效果，就不再是纯无监督评估。

In [5]:
print('模型       | 使用标签 | 指标/结果')  # 输出结果表头。
print(f'规则基线   | 否       | accuracy={baseline_accuracy:.3f}')  # 输出规则基线行。
print(f'逻辑回归   | 是       | accuracy={supervised_accuracy:.3f}')  # 输出监督学习行。
print('KMeans     | 否       | 两个运营画像簇')  # 输出无监督学习行。
print('解释：上述分数仅来自 8 条教学样本，不能外推到线上商户。')  # 明确受控实验的边界。

模型       | 使用标签 | 指标/结果
规则基线   | 否       | accuracy=1.000
逻辑回归   | 是       | accuracy=1.000
KMeans     | 否       | 两个运营画像簇
解释：上述分数仅来自 8 条教学样本，不能外推到线上商户。


## 失败案例与修复

错误做法是为了让聚类看起来更好，直接把超时标签拼进聚类特征。这样无监督过程偷看了标签，得到的簇当然更像风险分类。修复是将标签完全排除在 KMeans 输入之外，只在离线解释阶段单独核验。

In [6]:
leaky_features = np.c_[features, late_label * 100.0]  # 故意把标签放大后拼入聚类特征以构造泄漏。
leaky_distance = ((leaky_features[:, None, :] - leaky_features[[0, 3]][None, :, :]) ** 2).sum(axis=2)  # 计算泄漏特征空间中的距离。
leaky_cluster = leaky_distance.argmin(axis=1)  # 得到被标签主导的伪分群结果。
leaky_purity = max(np.mean(leaky_cluster == late_label), np.mean(1 - leaky_cluster == late_label))  # 计算允许簇编号翻转后的表面纯度。
clean_purity = max(np.mean(cluster == late_label), np.mean(1 - cluster == late_label))  # 计算未使用标签的真实事后纯度。
print(f'错误：标签泄漏后簇纯度={leaky_purity:.3f}')  # 展示泄漏带来的虚假漂亮指标。
print(f'修复：只输入画像后簇纯度={clean_purity:.3f}')  # 展示干净流程的真实结果。
print('生产差距：线上需按时间和商户切分、记录特征可见时间，并用人工画像与下游 A/B 评估分群。')  # 说明从教学实现到生产系统还缺失的能力。

错误：标签泄漏后簇纯度=1.000
修复：只输入画像后簇纯度=1.000
生产差距：线上需按时间和商户切分、记录特征可见时间，并用人工画像与下游 A/B 评估分群。


In [7]:
assert features.shape == (8, 2)  # 保护原始商户特征维度不被意外改变。
assert baseline_accuracy <= supervised_accuracy  # 保护监督模型至少不弱于该规则基线。
assert len(np.unique(cluster)) == 2  # 保护无监督实验确实得到两个非空簇。
assert leaky_purity >= clean_purity  # 保护标签泄漏会使表面指标虚高这一失败现象。